# Integración agrícola-climática con NASA POWER (2003–2025)

Este notebook une la producción agrícola con el clima municipal consolidado de NASA POWER. No descarga datos ni utiliza otras fuentes climáticas. La unidad climática es municipio-año.

In [8]:
import pandas as pd

from proyecto_agronomia.config import (
    NASA_POWER_PROCESSED_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
)
from proyecto_agronomia.processing.integracion_climatica import crear_clave_municipal

RUTA_AGRICULTURA = RAW_DATA_DIR / "agricultura"
RUTA_CLIMA_NASA = NASA_POWER_PROCESSED_DIR / "clima_municipal_2003_2025.csv"
RUTA_SALIDA = PROCESSED_DATA_DIR / "agricultura_clima_2003_2025.csv"
pd.set_option("display.max_columns", None)

## 1. Cargar la producción agrícola 2003–2025

In [9]:
archivos_agricultura = sorted(RUTA_AGRICULTURA.glob("agricola_*.csv"))
partes_agricultura = []

for archivo in archivos_agricultura:
    anio = int(archivo.stem.split("_")[-1])
    if 2003 <= anio <= 2025:
        datos = pd.read_csv(archivo, encoding="latin-1", low_memory=False)
        datos["Anio"] = anio
        partes_agricultura.append(datos)

if not partes_agricultura:
    raise FileNotFoundError(f"No se encontraron CSV agrícolas en {RUTA_AGRICULTURA}")

df_agricultura = pd.concat(partes_agricultura, ignore_index=True)
df_agricultura = crear_clave_municipal(df_agricultura)

assert df_agricultura["Anio"].between(2003, 2025).all()
print("Agricultura:", df_agricultura.shape)
display(df_agricultura.groupby("Anio").agg(
    registros=("cvegeo", "size"),
    municipios=("cvegeo", "nunique"),
    cultivos=("Idcultivo", "nunique"),
))

Agricultura: (825089, 27)


,registros,municipios,cultivos
Anio,,,
2003,35667,2410,307
2004,36204,2418,311
2005,36520,2418,324
2006,36056,2422,317
2007,36136,2428,326
2008,35785,2431,331
2009,35562,2433,314
2010,36228,2433,322
2011,36299,2432,329


## 2. Cargar y validar NASA POWER 2003–2025

In [10]:
if not RUTA_CLIMA_NASA.exists():
    raise FileNotFoundError(f"No existe el consolidado NASA POWER: {RUTA_CLIMA_NASA}")

columnas_nasa = [
    "cvegeo",
    "anio",
    "precipitacion_anual_mm",
    "temperatura_media_c",
]
df_clima_nasa = pd.read_csv(
    RUTA_CLIMA_NASA,
    usecols=columnas_nasa,
    dtype={"cvegeo": "string"},
)
df_clima_nasa["cvegeo"] = df_clima_nasa["cvegeo"].str.zfill(5)

claves_agricolas = set(df_agricultura["cvegeo"].unique())
claves_nasa = set(df_clima_nasa["cvegeo"].unique())
anios_esperados = set(range(2003, 2026))

assert not df_clima_nasa.duplicated(["cvegeo", "anio"]).any()
assert set(df_clima_nasa["anio"]) == anios_esperados
assert df_clima_nasa[["precipitacion_anual_mm", "temperatura_media_c"]].notna().all().all()
assert not (claves_agricolas - claves_nasa), (
    f"Municipios agrícolas sin NASA POWER: {sorted(claves_agricolas - claves_nasa)[:10]}"
)

print("Clima NASA:", df_clima_nasa.shape)
print("Municipios NASA:", df_clima_nasa["cvegeo"].nunique())
display(df_clima_nasa.groupby("anio").agg(
    municipios=("cvegeo", "nunique"),
    precipitacion_disponible=("precipitacion_anual_mm", "count"),
    temperatura_disponible=("temperatura_media_c", "count"),
))

Clima NASA: (56994, 4)
Municipios NASA: 2478


,municipios,precipitacion_disponible,temperatura_disponible
anio,,,
2003,2478,2478,2478
2004,2478,2478,2478
2005,2478,2478,2478
2006,2478,2478,2478
2007,2478,2478,2478
2008,2478,2478,2478
2009,2478,2478,2478
2010,2478,2478,2478
2011,2478,2478,2478


## 3. Unir agricultura y clima NASA

In [11]:
df_final = (
    df_agricultura.merge(
        df_clima_nasa,
        left_on=["cvegeo", "Anio"],
        right_on=["cvegeo", "anio"],
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

sin_clima = df_final["_merge"].ne("both")
if sin_clima.any():
    ejemplos = df_final.loc[sin_clima, ["cvegeo", "Anio"]].drop_duplicates().head(10)
    raise ValueError(f"Hay registros agrícolas sin clima NASA:\n{ejemplos}")

df_final = df_final.drop(columns=["anio", "_merge"])

assert len(df_final) == len(df_agricultura)
assert df_final[["precipitacion_anual_mm", "temperatura_media_c"]].notna().all().all()
print("Registros agrícolas originales:", len(df_agricultura))
print("Registros integrados:", len(df_final))
display(df_final[[
    "Anio", "cvegeo", "Nomestado", "Nommunicipio", "Nomcultivo",
    "precipitacion_anual_mm", "temperatura_media_c",
]].head())

Registros agrícolas originales: 825089
Registros integrados: 825089


,Anio,cvegeo,Nomestado,Nommunicipio,Nomcultivo,precipitacion_anual_mm,temperatura_media_c
0,2003,01001,Aguascalientes,Aguascalientes,Avena forrajera en verde,642.67,16.815123
1,2003,01001,Aguascalientes,Aguascalientes,Cebada forrajera en verde,642.67,16.815123
2,2003,01001,Aguascalientes,Aguascalientes,Centeno forrajero en verde,642.67,16.815123
3,2003,01001,Aguascalientes,Aguascalientes,Triticale forrajero en verde,642.67,16.815123
4,2003,01001,Aguascalientes,Aguascalientes,Pastos y praderas,642.67,16.815123


## 4. Validar cobertura climática

In [12]:
resumen_cobertura = df_final.groupby("Anio").agg(
    registros=("cvegeo", "size"),
    municipios=("cvegeo", "nunique"),
    con_precipitacion=("precipitacion_anual_mm", "count"),
    con_temperatura=("temperatura_media_c", "count"),
)
resumen_cobertura["cobertura_precipitacion_pct"] = (
    100 * resumen_cobertura["con_precipitacion"] / resumen_cobertura["registros"]
).round(2)
resumen_cobertura["cobertura_temperatura_pct"] = (
    100 * resumen_cobertura["con_temperatura"] / resumen_cobertura["registros"]
).round(2)
display(resumen_cobertura)
assert resumen_cobertura[["cobertura_precipitacion_pct", "cobertura_temperatura_pct"]].eq(100).all().all()

,registros,municipios,con_precipitacion,con_temperatura,cobertura_precipitacion_pct,cobertura_temperatura_pct
Anio,,,,,,
2003,35667,2410,35667,35667,100.0,100.0
2004,36204,2418,36204,36204,100.0,100.0
2005,36520,2418,36520,36520,100.0,100.0
2006,36056,2422,36056,36056,100.0,100.0
2007,36136,2428,36136,36136,100.0,100.0
2008,35785,2431,35785,35785,100.0,100.0
2009,35562,2433,35562,35562,100.0,100.0
2010,36228,2433,36228,36228,100.0,100.0
2011,36299,2432,36299,36299,100.0,100.0


## 5. Guardar el dataset integrado

In [13]:
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(RUTA_SALIDA, index=False, encoding="utf-8-sig")
print("Dataset guardado:", RUTA_SALIDA)
print("Dimensiones:", df_final.shape)

Dataset guardado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_2003_2025.csv
Dimensiones: (825089, 29)


## 6. Cargar y revisar el CSV generado

In [14]:
df_agricultura_clima = pd.read_csv(
    RUTA_SALIDA,
    dtype={"cvegeo": "string"},
    low_memory=False,
)
df_agricultura_clima["cvegeo"] = df_agricultura_clima["cvegeo"].str.zfill(5)

print("Archivo:", RUTA_SALIDA)
print("Dimensiones:", df_agricultura_clima.shape)
print("Periodo:", df_agricultura_clima["Anio"].min(), "-", df_agricultura_clima["Anio"].max())
print("Municipios:", df_agricultura_clima["cvegeo"].nunique())
display(df_agricultura_clima.head())

Archivo: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_2003_2025.csv
Dimensiones: (825089, 29)
Periodo: 2003 - 2025
Municipios: 2452


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,Nomcicloproductivo,Idmodalidad,Nommodalidad,Idunidadmedida,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion,Nomcultivo Sin Um,Preciomediorural,cvegeo,precipitacion_anual_mm,temperatura_media_c
0,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5490000,Avena forrajera en verde,1062.0,1062.0,0.0,28529.0,26.86,323.4,9226278.6,NaN,NaN,01001,642.67,16.815123
1,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5900000,Cebada forrajera en verde,2.0,2.0,0.0,54.0,27.0,250.0,13500.0,NaN,NaN,01001,642.67,16.815123
2,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5980000,Centeno forrajero en verde,17.0,17.0,0.0,1020.0,60.0,350.0,357000.0,NaN,NaN,01001,642.67,16.815123
3,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,9090000,Triticale forrajero en verde,24.0,24.0,0.0,720.0,30.0,400.0,288000.0,NaN,NaN,01001,642.67,16.815123
4,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,15050000,Pastos y praderas,415.0,415.0,0.0,25830.0,62.24,314.9,8133867.0,NaN,NaN,01001,642.67,16.815123
